# AEMO offline RL workflow notebook

This notebook replaces the script-first AEMO workflow with an inspectable notebook flow for:

1. fetching + caching AEMO market data across multiple regions and time windows
2. sweeping multiple battery sizes
3. collecting rule / dispatch-replay / SB3 trajectories
4. exporting parquet logs and a DT-ready dataset
5. optionally launching Decision Transformer training

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import polars as pl

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

In [ ]:
from aemo_notebook_utils import (
    build_dispatch_selection,
    build_dt_dataset_from_logs,
    build_model_config,
    fetch_and_preprocess_aemo_scenarios,
    fit_aemo_global_stats,
    launch_dt_training,
    prepare_run_paths,
    resolve_battery_variants,
    run_rule_episodes,
    run_sb3_episodes,
    validate_aemo_dt_dimensions,
    write_combined_episode_logs,
    write_json,
)
from dispatch_utils import run_dispatch_replay

## 1. Experiment configuration

Edit this cell first. The notebook is designed so you can inspect each intermediate object before moving to the next stage.

In [ ]:
SCENARIOS = [
    {
        'label': 'sa1_2022_2023',
        'region': 'SA1',
        'start_date': datetime.fromisoformat('2022-01-01'),
        'end_date': datetime.fromisoformat('2023-03-01'),
    },
    {
        'label': 'vic1_2021_2022',
        'region': 'VIC1',
        'start_date': datetime.fromisoformat('2021-01-01'),
        'end_date': datetime.fromisoformat('2022-03-01'),
    },
]

STEP_DURATION = 5 / 60
EPISODE_HOURS = 24 * 5
ACTION_MODE = 'multi_market'
DEGRADATION_MODE = 'real_world'
DEGRADATION_CHEMISTRY = 'LFP'
DEGRADATION_TEMPERATURE = 30.0
CONTEXT_LENGTH = 288
DATASET_TAG = 'aemo_dt'

CACHE_DIR = REPO_ROOT / 'src' / 'data' / 'aemo'
OUTPUT_DIR = REPO_ROOT / 'data' / 'aemo_dt'
MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'aemo_decision_transformer_model_kwargs.json'
SCENARIO_MANIFEST_PATH = OUTPUT_DIR / f'{DATASET_TAG}_scenario_manifest.json'

BATTERY_VARIANTS = [
    {'name': 'small', 'capacity_mwh': 2.0, 'max_power_mw': 1.0, 'init_soc_ratio': 0.5},
    {'name': 'medium', 'capacity_mwh': 10.0, 'max_power_mw': 5.0, 'init_soc_ratio': 0.5},
    {'name': 'large', 'capacity_mwh': 50.0, 'max_power_mw': 25.0, 'init_soc_ratio': 0.5},
]

BEHAVIOR_RUNS = [
    {
        'policy': 'rule',
        'episodes': 4,
        'battery_variants': ['small', 'medium', 'large'],
        'random_episode_start': True,
        'seed': 42,
    },
    {
        'policy': 'dispatch',
        'episodes': 2,
        'battery_variants': ['medium'],
        'station_name': 'hornsdale',
    },
    {
        'policy': 'sb3',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'PPO',
        'model_path': REPO_ROOT / 'models' / 'aemo_ppo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
]

RUN_DT_TRAINING = False
DT_TRAINING_ARGS = {
    'epochs': 5,
    'batch_size': 8,
    'lr': 2e-5,
    'val_split': 0.1,
    'seed': 42,
    'device': None,
    'amp_mode': 'off',
    'return_scale': 1.0,
    'action_loss_weight': 1.0,
    'state_loss_weight': 0.01,
    'return_loss_weight': 0.002,
    'weight_decay': 1e-4,
    'num_workers': 2,
    'prefetch_factor': 2,
}

## 2. Fetch and cache multi-scenario AEMO data

In [ ]:
run_paths = prepare_run_paths(output_dir=OUTPUT_DIR, dataset_tag=DATASET_TAG)

global_stats, scenario_manifest = fit_aemo_global_stats(
    scenarios=SCENARIOS,
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION,
    refresh=False,
)
processed_by_label, _ = fetch_and_preprocess_aemo_scenarios(
    scenarios=SCENARIOS,
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION,
    refresh=False,
    fixed_stats=global_stats,
)
scenario_manifest_payload = [
    {
        **entry,
        'start_date': entry['start_date'].isoformat(),
        'end_date': entry['end_date'].isoformat(),
    }
    for entry in scenario_manifest
]
scenario_lookup = {entry['label']: entry for entry in scenario_manifest}
scenario_payloads = [
    (scenario_lookup[entry['label']], processed_by_label[entry['label']])
    for entry in scenario_manifest
]
resolved_battery_variants = resolve_battery_variants(BATTERY_VARIANTS)
MAX_STEP = int(round(EPISODE_HOURS / STEP_DURATION))
MAX_TIMESTEP = MAX_STEP

write_json(
    SCENARIO_MANIFEST_PATH,
    {
        'global_stats': global_stats,
        'scenarios': scenario_manifest_payload,
    },
)

print(f'scenario count: {len(scenario_manifest)}')
print(f'regions: {sorted({entry["region"] for entry in scenario_manifest})}')
print(f'processed cache dir: {CACHE_DIR}')
pl.DataFrame(scenario_manifest_payload)

In [ ]:
processed_data.head()

## 3. Collect behavior-policy trajectories per scenario

Each run is tagged by `scenario__policy__battery_label` so you can compare scenarios, policies, and battery sizes independently.

In [ ]:
all_logs = {}
raw_outputs = {}

for scenario_entry, scenario_processed_data in scenario_payloads:
    for run in BEHAVIOR_RUNS:
        selected_labels = set(run.get('battery_variants', [variant['label'] for variant in resolved_battery_variants]))
        selected_variants = [variant for variant in resolved_battery_variants if variant['label'] in selected_labels]

        for variant in selected_variants:
            tag = f"{scenario_entry['label']}__{run['policy']}__{variant['label']}"
            print(f'Collecting {tag}...')

            if run['policy'] == 'rule':
                episodes = run_rule_episodes(
                    processed_data=scenario_processed_data,
                    num_episodes=run['episodes'],
                    battery_capacity=variant['battery_capacity'],
                    max_battery_flow=variant['max_battery_flow'],
                    init_soc=variant['init_soc'],
                    step_duration=STEP_DURATION,
                    battery_life_cost=variant['battery_life_cost'],
                    max_step=MAX_STEP,
                    action_mode=ACTION_MODE,
                    degradation_mode=DEGRADATION_MODE,
                    degradation_chemistry=DEGRADATION_CHEMISTRY,
                    degradation_temperature=DEGRADATION_TEMPERATURE,
                    random_episode_start=run.get('random_episode_start', True),
                    base_seed=run.get('seed', 42),
                )
            elif run['policy'] == 'dispatch':
                selection = build_dispatch_selection(
                    region=scenario_entry['region'],
                    start_date=scenario_entry['start_date'],
                    end_date=scenario_entry['end_date'],
                    cache_dir=CACHE_DIR,
                    dispatch_station=run.get('station_name'),
                    dispatch_duid=run.get('dispatch_duid'),
                    dispatch_index=run.get('dispatch_index', 0),
                    battery_capacity=variant['battery_capacity'],
                    max_battery_flow=variant['max_battery_flow'],
                    init_soc=variant['init_soc'],
                )
                episodes, incident_logs, _ = run_dispatch_replay(
                    processed_data=scenario_processed_data,
                    selection=selection,
                    start_date=scenario_entry['start_date'],
                    end_date=scenario_entry['end_date'],
                    region=scenario_entry['region'],
                    cache_dir=str(CACHE_DIR),
                    num_episodes=run['episodes'],
                    step_duration=STEP_DURATION,
                    battery_life_cost=variant['battery_life_cost'],
                    max_step=MAX_STEP,
                    output_dir=None,
                    run_tag=tag,
                    action_mode=ACTION_MODE,
                    degradation_mode=DEGRADATION_MODE,
                    degradation_chemistry=DEGRADATION_CHEMISTRY,
                    degradation_temperature=DEGRADATION_TEMPERATURE,
                )
                if any(df.height > 0 for df in incident_logs):
                    incident_path = run_paths['raw_dir'] / f'{tag}_incident_logs.parquet'
                    pl.concat([df.with_columns(pl.lit(i).alias('episode_id')) for i, df in enumerate(incident_logs) if df.height > 0], how='diagonal_relaxed').write_parquet(incident_path)
                    raw_outputs[f'{tag}__incidents'] = str(incident_path)
            elif run['policy'] == 'sb3':
                episodes = run_sb3_episodes(
                    processed_data=scenario_processed_data,
                    battery_variant=variant,
                    model_path=run['model_path'],
                    algorithm=run['algorithm'],
                    num_episodes=run['episodes'],
                    max_step=MAX_STEP,
                    step_duration=STEP_DURATION,
                    action_mode=ACTION_MODE,
                    degradation_mode=DEGRADATION_MODE,
                    degradation_chemistry=DEGRADATION_CHEMISTRY,
                    degradation_temperature=DEGRADATION_TEMPERATURE,
                    random_episode_start=run.get('random_episode_start', True),
                    deterministic=run.get('deterministic', True),
                )
            else:
                raise ValueError(f"Unsupported policy: {run['policy']}")

            tagged_episodes = [
                episode.with_columns(
                    pl.lit(scenario_entry['label']).alias('scenario_label'),
                    pl.lit(scenario_entry['region']).alias('scenario_region'),
                    pl.lit(scenario_entry['start_date'].isoformat()).alias('scenario_start_date'),
                    pl.lit(scenario_entry['end_date'].isoformat()).alias('scenario_end_date'),
                    pl.lit(run['policy']).alias('policy_name'),
                    pl.lit(variant['label']).alias('battery_label'),
                )
                for episode in episodes
            ]
            raw_path = run_paths['raw_dir'] / f'{tag}_logs.parquet'
            write_combined_episode_logs(episodes=tagged_episodes, output_path=raw_path)
            all_logs[tag] = tagged_episodes
            raw_outputs[tag] = str(raw_path)

print(sorted(all_logs))

## 4. Build the DT dataset and manifest

The dataset manifest now records the scenario list, shared normalization stats, and the per-run raw log outputs.

In [ ]:
dataset, manifest = build_dt_dataset_from_logs(all_logs)
validate_aemo_dt_dimensions(manifest, action_mode=ACTION_MODE)
model_kwargs = build_model_config(
    action_mode=ACTION_MODE,
    context_len=CONTEXT_LENGTH,
    max_timestep=MAX_TIMESTEP,
    output_path=MODEL_CONFIG_PATH,
)

dataset.write_parquet(run_paths['dataset_path'])
manifest.update({
    'created_at': datetime.now(timezone.utc).isoformat(),
    'dataset_path': str(run_paths['dataset_path']),
    'manifest_path': str(run_paths['manifest_path']),
    'scenario_manifest_path': str(SCENARIO_MANIFEST_PATH),
    'cache_dir': str(CACHE_DIR),
    'region_count': len({entry['region'] for entry in scenario_manifest}),
    'scenario_count': len(scenario_manifest),
    'global_stats': global_stats,
    'scenarios': scenario_manifest_payload,
    'step_duration': STEP_DURATION,
    'episode_hours': EPISODE_HOURS,
    'max_step': MAX_STEP,
    'action_mode': ACTION_MODE,
    'degradation_mode': DEGRADATION_MODE,
    'degradation_chemistry': DEGRADATION_CHEMISTRY,
    'degradation_temperature': DEGRADATION_TEMPERATURE,
    'battery_variants': resolved_battery_variants,
    'behavior_runs': BEHAVIOR_RUNS,
    'model_config_path': str(MODEL_CONFIG_PATH),
    'model_kwargs': model_kwargs,
    'raw_outputs': raw_outputs,
})
write_json(run_paths['manifest_path'], manifest)

print(run_paths['dataset_path'])
print(run_paths['manifest_path'])
dataset.head()

## 5. Optional: launch Decision Transformer training

Set `RUN_DT_TRAINING = True` in the config cell if you want the notebook to call the trainer.

In [ ]:
if RUN_DT_TRAINING:
    save_path = run_paths['output_dir'] / f'{DATASET_TAG}_dt_model.pt'
    checkpoint_path = run_paths['output_dir'] / f'{DATASET_TAG}_dt_checkpoint.pt'
    loss_csv_path = run_paths['output_dir'] / f'{DATASET_TAG}_dt_loss_history.csv'
    command = launch_dt_training(
        dataset_path=run_paths['dataset_path'],
        model_config_path=MODEL_CONFIG_PATH,
        save_path=save_path,
        checkpoint_path=checkpoint_path,
        loss_csv_path=loss_csv_path,
        epochs=DT_TRAINING_ARGS['epochs'],
        batch_size=DT_TRAINING_ARGS['batch_size'],
        lr=DT_TRAINING_ARGS['lr'],
        val_split=DT_TRAINING_ARGS['val_split'],
        seed=DT_TRAINING_ARGS['seed'],
        device=DT_TRAINING_ARGS['device'],
        amp_mode=DT_TRAINING_ARGS['amp_mode'],
        return_scale=DT_TRAINING_ARGS['return_scale'],
        action_loss_weight=DT_TRAINING_ARGS['action_loss_weight'],
        state_loss_weight=DT_TRAINING_ARGS['state_loss_weight'],
        return_loss_weight=DT_TRAINING_ARGS['return_loss_weight'],
        weight_decay=DT_TRAINING_ARGS['weight_decay'],
        num_workers=DT_TRAINING_ARGS['num_workers'],
        prefetch_factor=DT_TRAINING_ARGS['prefetch_factor'],
    )
    print(' '.join(command))
else:
    print('DT training skipped. Set RUN_DT_TRAINING = True to enable it.')